# 意识知觉的脑电标记：看见 vs 没看见（N = 31）—— 最终分析笔记本
本笔记本是论文中**所有统计结果的唯一来源**：从 QC 和剔除标准开始，依次运行组水平计算、统计与作图、稳健性检验、时间分块交叉验证，最后用一个核对单元打印论文中的每个关键数字。

**前置步骤**（在终端中运行，见 README）：`code/01_preprocess.py`（逐被试）→ `code/01b_make_compact.py`。

| 节 | 内容 | 对应脚本 |
|---|---|---|
| 0 | 路径设置 | — |
| 1 | QC 表、剔除标准、行为；逐被试 ERP / 时频 / 解码 | `02_group_compute.py` |
| 2 | ERP 时空聚类、ROI 检验（含 Bonferroni）、时频聚类、解码聚类、时间泛化、刺激前 alpha、图 1–4 | `03_group_stats_figures.py` |
| 3 | 稳健性：时间泛化显著范围；100 次对比度匹配；回归（看见 + 对比度 + 试次序号）；贝叶斯因子 | `05_robustness_checks.py` |
| 4 | 时间分块交叉验证 | `06_blocked_cv.py` |
| 5 | 核对单元：二维聚类的连通性检查 + 论文数字对照 | — |

In [1]:
# 0. 路径：数据在 derivatives/compact/，所有输出写入 output/
import os, warnings; warnings.filterwarnings("ignore")
ROOT = os.path.abspath(".")
D = os.path.join(ROOT, "derivatives")
if not os.path.isdir(os.path.join(D, "compact")):          # 作者分析环境中的数据位置
    D = "/mnt/user-data/uploads/Downloads/EEG_consciousness/derivatives"
OUT = os.path.join(ROOT, "output"); os.makedirs(os.path.join(OUT, "figs"), exist_ok=True); os.chdir(OUT)
print("data:", D, "| output:", OUT)

data: /mnt/user-data/uploads/Downloads/EEG_consciousness/derivatives | output: /home/claude/repo/eeg-awareness-ds006171/output


## 1. QC、剔除标准与逐被试计算（约 5–10 分钟）
**剔除标准（预先设定）**：插值导联 > 10%；任一条件有效试次 < 30；捕获试次虚报率 > 30%。下方会打印逐被试的 QC 表（插值导联数、看见/没看见试次数、虚报率、剔除原因、是否纳入），并保存为 `results_behavior_qc.csv`。
- **解码**：标准化与逻辑回归组成同一个 pipeline → 标准化参数只在训练折上估计；分层 5 折、不打乱；不调超参数（C = 1）。

In [2]:
# 组水平分析：读取 QC、应用剔除标准、行为数据
import mne, numpy as np, pandas as pd, glob, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy import stats
from sklearn.pipeline import make_pipeline; from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from mne.decoding import SlidingEstimator, GeneralizingEstimator, cross_val_multiscore
mne.set_log_level("ERROR"); rng = 0
qc = pd.read_csv(f"{D}/qc_all.csv")
# ---------- 剔除标准（看结果之前预先设定）----------
qc["n_interp"] = qc.bads_post_ica.fillna("").apply(lambda s: len([c for c in s.split(";") if c]))
rows = []
import os
for s in qc["sub"]:
    f = f"{D}/compact/{s}_task-nocue_desc-128Hz_epo.fif"
    if not os.path.exists(f): rows.append(dict(sub=s)); continue      # 预处理质量已判定不合格、未传输的被试
    md = mne.read_epochs(f"{D}/compact/{s}_task-nocue_desc-128Hz_epo.fif", preload=False).metadata
    rows.append(dict(sub=s, n_seen=(md.condition=="seen").sum(), n_unseen=(md.condition=="unseen").sum(),
                     hit_rate=(md.query("stimulus=='present'").seen==1).mean(), fa_rate=md.query("stimulus=='absent'").seen.mean(),
                     contrast_seen=md.query("condition=='seen'").contrast.mean(), contrast_unseen=md.query("condition=='unseen'").contrast.mean()))
beh = qc.merge(pd.DataFrame(rows), on="sub")
beh["exclude_reason"] = ""
beh.loc[beh.n_interp > 12.8, "exclude_reason"] += "interp>10%;"
beh.loc[(beh.n_seen < 30) | (beh.n_unseen < 30), "exclude_reason"] += "<30 trials;"
beh.loc[beh.fa_rate > 0.3, "exclude_reason"] += "FA>30%;"
print(beh.loc[beh.exclude_reason != "", ["sub", "n_interp", "n_seen", "n_unseen", "fa_rate", "exclude_reason"]].to_string(index=False))
subs = beh.loc[beh.exclude_reason == "", "sub"].tolist(); N = len(subs); print("N included =", N)
beh["included"] = beh.exclude_reason == ""
beh.to_csv("results_behavior_qc.csv", index=False)
print(beh[["sub","n_interp","n_seen","n_unseen","fa_rate","exclude_reason","included"]].to_string(index=False))
b = beh[beh.exclude_reason == ""]
t, p = stats.ttest_rel(b.contrast_seen, b.contrast_unseen)
print("Behavior: hit rate %.2f±%.2f, FA %.3f±%.3f; contrast seen %.4f vs unseen %.4f (diff %.1f%%), t(%d)=%.2f p=%.3g" %
      (b.hit_rate.mean(), b.hit_rate.std(), b.fa_rate.mean(), b.fa_rate.std(), b.contrast_seen.mean(), b.contrast_unseen.mean(),
       100*(b.contrast_seen.mean()/b.contrast_unseen.mean()-1), N-1, t, p))

# ---------- 逐被试：ERP、时频、解码 ----------
freqs = np.arange(4, 31, 1); post = ["CPz","CPPz","Pz","P1","P2","PO3","PO4","POz","Oz","O1","O2"]
ERP = {"seen": [], "unseen": [], "catch": []}; TF = {"seen": [], "unseen": []}; PRE = []; AUC = []; GEN = []
for s in subs:
    e = mne.read_epochs(f"{D}/compact/{s}_task-nocue_desc-128Hz_epo.fif")
    ec = e.copy().crop(-0.2, 1.0)
    ERP["seen"].append(ec["condition=='seen'"].average()); ERP["unseen"].append(ec["condition=='unseen'"].average())
    ERP["catch"].append(ec["stimulus=='absent'"].average())
    ep = e["stimulus=='present'"]; chs = [c for c in post if c in e.ch_names]; pre = {}
    for cond in ["seen", "unseen"]:
        p_ = ep[f"condition=='{cond}'"].copy().pick(chs).compute_tfr("morlet", freqs=freqs, n_cycles=freqs/2, decim=2, average=True, return_itc=False)
        pre[cond] = 10*np.log10(p_.copy().crop(-0.5, -0.1).data.mean(0)[(freqs>=8)&(freqs<=12)].mean() * 1e12)
        p_.apply_baseline((-0.5, -0.2), mode="logratio"); p_.crop(-0.3, 1.0); TF[cond].append(p_.data.mean(0)*10)
    tf_times = p_.times; PRE.append(pre)
    ed = ep.copy().crop(-0.2, 1.0).resample(64); X = ed.get_data(); y = (ed.metadata.condition=="seen").values.astype(int)
    clf = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
    AUC.append(cross_val_multiscore(SlidingEstimator(clf, scoring="roc_auc", n_jobs=8), X, y, cv=5).mean(0))
    GEN.append(cross_val_multiscore(GeneralizingEstimator(clf, scoring="roc_auc", n_jobs=8), X, y, cv=5).mean(0))
    dec_times = ed.times; print(s, "done", flush=True)
np.savez("group_data.npz", AUC=np.array(AUC), GEN=np.array(GEN), TFs=np.array(TF["seen"]), TFu=np.array(TF["unseen"]), tf_times=tf_times, dec_times=dec_times, freqs=freqs)
for k in ERP: mne.write_evokeds(f"erp_{k}-ave.fif", ERP[k], overwrite=True)
pd.DataFrame([dict(sub=s, **p) for s, p in zip(subs, PRE)]).to_csv("results_prestim_alpha.csv", index=False)
print("ALL SUBJECTS DONE")


   sub  n_interp  n_seen  n_unseen  fa_rate exclude_reason
sub-08        42     NaN       NaN      NaN    interp>10%;
sub-16         0     7.0      17.0      0.0    <30 trials;
sub-20         0    23.0      28.0      0.0    <30 trials;
sub-34        16     NaN       NaN      NaN    interp>10%;
N included = 31
   sub  n_interp  n_seen  n_unseen  fa_rate exclude_reason  included
sub-01         1   155.0     131.0 0.000000                     True
sub-02         1   105.0     191.0 0.000000                     True
sub-03         0   134.0     123.0 0.100000                     True
sub-04         0   150.0     206.0 0.025000                     True
sub-05         8   103.0     120.0 0.142857                     True
sub-06         0   176.0     184.0 0.025000                     True
sub-07         0   156.0     189.0 0.000000                     True
sub-08        42     NaN       NaN      NaN    interp>10%;     False
sub-09         0   183.0     176.0 0.150000                     True

sub-01 done


sub-02 done


sub-03 done


sub-04 done


...

sub-31 done


sub-32 done


sub-33 done


sub-36 done


ALL SUBJECTS DONE


## 2. 组水平统计与图 1–4
- **ERP 时空聚类**：全部 128 导 × 0–800 ms，簇形成阈值 p < .01（双尾），2000 次置换，导联邻接来自电极位置。
- **ROI 检验**：VAN、P3b 两个检验，同时输出原始 p 与 Bonferroni 校正 p（×2）。
- **时频聚类**：被试 × 频率 × 时间，二维网格邻接（频率与时间均相邻），2000 次置换。
- **解码 / 时间泛化**：对每位被试的 AUC − 0.5 做符号翻转的单侧聚类置换检验（2000 / 1000 次）。

In [3]:
# 组水平统计与作图
import mne, numpy as np, pandas as pd, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy import stats
mne.set_log_level("ERROR")
ERP = {k: mne.read_evokeds(f"erp_{k}-ave.fif") for k in ["seen", "unseen", "catch"]}; N = len(ERP["seen"])
G = np.load("group_data.npz"); out = {}
info = ERP["seen"][0].info; times = ERP["seen"][0].times; tm = times*1000
roi = {"P3b": ["CPz","CPPz","Pz","P1","P2"], "VAN": ["PO7","PO8","P7","P8","O1","O2"]}

# ---- (1) ERP 时空聚类置换检验：全部 128 导 × 0–800 ms，看见 vs 没看见
X = np.array([s.data - u.data for s, u in zip(ERP["seen"], ERP["unseen"])])        # 被试 × 导联 × 时间
tsel = (times >= 0) & (times <= 0.8); Xc = X[:, :, tsel].transpose(0, 2, 1)          # → 被试 × 时间 × 导联
adj, _ = mne.channels.find_ch_adjacency(info, "eeg")
T, clu, cp, _ = mne.stats.spatio_temporal_cluster_1samp_test(Xc, adjacency=adj, n_permutations=2000, tail=0, seed=0, n_jobs=8,
                                                            threshold=stats.t.ppf(0.995, N-1))   # 簇形成阈值 p<0.01
clus = []
for c, p in zip(clu, cp):
    if p < 0.05:
        ti, ci = c; tt = tm[tsel][np.unique(ti)]; sign = np.sign(T[ti, ci].mean())
        clus.append(dict(sign="positive" if sign > 0 else "negative", start_ms=round(tt.min()), end_ms=round(tt.max()),
                         n_channels=len(np.unique(ci)), p=round(p, 4),
                         channels=",".join(np.array(info.ch_names)[np.unique(ci)][:12])))
clus = pd.DataFrame(clus).sort_values("start_ms"); print("ERP spatio-temporal clusters (seen − unseen):\n", clus.to_string(index=False)); clus.to_csv("results_erp_clusters.csv", index=False)

# ---- (2) ROI 平均幅值
def amp(evs, chs, t0, t1): return np.array([e.copy().pick(chs).crop(t0, t1).data.mean()*1e6 for e in evs])
rr = []
for name, chs, w in [("VAN (occipito-temporal)", roi["VAN"], (0.18, 0.28)), ("P3b (centro-parietal)", roi["P3b"], (0.35, 0.65))]:
    a, u = amp(ERP["seen"], chs, *w), amp(ERP["unseen"], chs, *w); t, p = stats.ttest_rel(a, u); d = (a-u).mean()/(a-u).std(ddof=1)
    rr.append([name, f"{int(w[0]*1000)}-{int(w[1]*1000)} ms", a.mean(), u.mean(), t, p, d, int(((a-u) > 0).sum() if "P3b" in name else ((a-u) < 0).sum())])
rr = pd.DataFrame(rr, columns=["component", "window", "seen µV", "unseen µV", f"t({N-1})", "p", "Cohen's dz", f"subjects in expected direction /{N}"]).round(4)
rr["p (Bonferroni, 2 tests)"] = np.minimum(rr["p"] * 2, 1).round(4)
print(rr.to_string(index=False)); rr.to_csv("results_erp_roi.csv", index=False)

# 图 1：ROI 波形
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (name, chs), win in zip(axs, [("Occipito-temporal (VAN)", roi["VAN"]), ("Centro-parietal (P3b)", roi["P3b"])], [(180, 280), (350, 650)]):
    for cond, col in [("seen", "#c0504d"), ("unseen", "#4f81bd"), ("catch", "#999999")]:
        sj = np.array([e.copy().pick(chs).data.mean(0) for e in ERP[cond]])*1e6; m, se = sj.mean(0), sj.std(0, ddof=1)/np.sqrt(N)
        ax.plot(tm, m, color=col, label=cond if cond != "catch" else "catch (no stimulus)"); ax.fill_between(tm, m-se, m+se, color=col, alpha=.2)
    ax.axvspan(*win, color="orange", alpha=.1); ax.axvline(0, c="k", lw=.6); ax.axhline(0, c="k", lw=.6)
    ax.set_title(f"{name}, N = {N}"); ax.set_xlabel("Time (ms)"); ax.set_ylabel("µV"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("figs/fig1_erp_roi.png", dpi=300); plt.close()
# 图 2：差异波地形图
diff = mne.combine_evoked([mne.grand_average(ERP["seen"]), mne.grand_average(ERP["unseen"])], weights=[1, -1])
f = diff.plot_topomap(times=[0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7], ch_type="eeg", show=False, time_unit="ms", vlim=(-2.5, 2.5))
f.suptitle(f"Seen − Unseen (µV), N = {N}"); f.savefig("figs/fig2_topo_diff.png", dpi=300); plt.close()
# 图 3：聚类时间-导联图
fig, ax = plt.subplots(figsize=(8, 4.5)); Tm = T.T.copy(); mask = np.zeros_like(Tm, bool)
for c, p in zip(clu, cp):
    if p < 0.05: mask[c[1], c[0]] = True
order = np.argsort([info["chs"][i]["loc"][1] for i in range(len(info.ch_names))])[::-1]           # 前→后排序
im = ax.imshow(np.where(mask, Tm, np.nan)[order], aspect="auto", cmap="RdBu_r", vmin=-8, vmax=8, extent=[0, 800, 128, 0])
ax.imshow(np.where(mask, np.nan, Tm)[order], aspect="auto", cmap="Greys", vmin=-8, vmax=8, alpha=.25, extent=[0, 800, 128, 0])
ax.set_xlabel("Time (ms)"); ax.set_ylabel("Channels (anterior → posterior)"); ax.set_title("Significant clusters (t-values), seen − unseen"); plt.colorbar(im, ax=ax, label="t")
plt.tight_layout(); plt.savefig("figs/fig3_clusters.png", dpi=300); plt.close()

# ---- (3) 时频：时间×频率聚类置换检验
tf_t, fr = G["tf_times"], G["freqs"]; Dtf = G["TFs"] - G["TFu"]
Ttf, clu_tf, cp_tf, _ = mne.stats.permutation_cluster_1samp_test(Dtf, n_permutations=2000, tail=0, seed=0, threshold=stats.t.ppf(0.995, N-1), n_jobs=8, out_type="mask")
tfc = []
for c, p in zip(clu_tf, cp_tf):
    if p < 0.05:
        fi, ti = np.where(c); tfc.append(dict(sign="+" if Ttf[c].mean() > 0 else "−", freq=f"{fr[fi].min()}-{fr[fi].max()} Hz", time=f"{round(tf_t[ti].min()*1000)}-{round(tf_t[ti].max()*1000)} ms", p=round(p, 4)))
tfc = pd.DataFrame(tfc); print("TF clusters:\n", tfc.to_string(index=False)); tfc.to_csv("results_tf_clusters.csv", index=False)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.7)); ext = [tf_t[0]*1000, tf_t[-1]*1000, fr[0], fr[-1]]
for a, dat, ttl, v in [(ax[0], G["TFs"].mean(0), "Seen", 2.5), (ax[1], G["TFu"].mean(0), "Unseen", 2.5), (ax[2], Dtf.mean(0), "Seen − Unseen", 1.0)]:
    im = a.imshow(dat, aspect="auto", origin="lower", extent=ext, cmap="RdBu_r", vmin=-v, vmax=v); plt.colorbar(im, ax=a, label="dB")
    a.axvline(0, c="k", lw=.6); a.set_title(f"{ttl} (posterior ROI)"); a.set_xlabel("Time (ms)"); a.set_ylabel("Hz")
sigm = np.zeros_like(Ttf, bool)
for c, p in zip(clu_tf, cp_tf):
    if p < 0.05: sigm |= c
if sigm.any(): ax[2].contour(sigm, levels=[.5], colors="k", linewidths=1, extent=ext, origin="lower")
plt.tight_layout(); plt.savefig("figs/fig4_tfr.png", dpi=300); plt.close()

# ---- (4) 解码：时间聚类置换检验 + 时间泛化
A = G["AUC"]; dt = G["dec_times"]*1000; m, se = A.mean(0), A.std(0, ddof=1)/np.sqrt(N)
Ta, clu_a, cp_a, _ = mne.stats.permutation_cluster_1samp_test(A - .5, n_permutations=2000, tail=1, seed=0, out_type="mask")
sig = np.zeros(len(dt), bool); dc = []
def to_mask(c, n):
    m_ = np.zeros(n, bool); m_[c if not isinstance(c, tuple) else c[0]] = True; return m_
for c, p in zip(clu_a, cp_a):
    c = to_mask(c, len(dt))
    if p < 0.05: sig |= c; dc.append(dict(start_ms=round(dt[c][0]), end_ms=round(dt[c][-1]), p=round(p, 4)))
dc = pd.DataFrame(dc); print("Decoding clusters:\n", dc.to_string(index=False)); dc.to_csv("results_decoding_clusters.csv", index=False)
print("Decoding peak AUC %.3f ± %.3f at %d ms" % (m.max(), se[m.argmax()], dt[m.argmax()]))
GEN = G["GEN"].mean(0)
Tg, clu_g, cp_g, _ = mne.stats.permutation_cluster_1samp_test(G["GEN"] - .5, n_permutations=1000, tail=1, seed=0, out_type="mask")
gm = np.zeros_like(GEN, bool)
for c, p in zip(clu_g, cp_g):
    if p < 0.05:
        if isinstance(c, tuple): mm = np.zeros_like(GEN, bool); mm[c] = True; c = mm
        gm |= c
fig, ax = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1.4, 1]})
ax[0].plot(dt, m, color="#c0504d", lw=2); ax[0].fill_between(dt, m-se, m+se, color="#c0504d", alpha=.25)
ax[0].axhline(.5, c="k", ls="--", lw=.7); ax[0].axvline(0, c="k", lw=.6)
if sig.any(): ax[0].fill_between(dt, .47, .48, where=sig, color="k", step="mid", label="cluster p < 0.05")
ax[0].set_xlabel("Time (ms)"); ax[0].set_ylabel("AUC"); ax[0].set_title(f"Decoding seen vs unseen (N = {N})"); ax[0].legend(fontsize=8)
im = ax[1].imshow(GEN, origin="lower", extent=[dt[0], dt[-1], dt[0], dt[-1]], cmap="RdBu_r", vmin=.4, vmax=.6)
if gm.any(): ax[1].contour(gm, levels=[.5], colors="k", linewidths=.8, extent=[dt[0], dt[-1], dt[0], dt[-1]], origin="lower")
ax[1].set_xlabel("Testing time (ms)"); ax[1].set_ylabel("Training time (ms)"); ax[1].set_title("Temporal generalization"); plt.colorbar(im, ax=ax[1], label="AUC")
plt.tight_layout(); plt.savefig("figs/fig5_decoding.png", dpi=300); plt.close()

# ---- (5) 刺激前 alpha
pre = pd.read_csv("results_prestim_alpha.csv"); t, p = stats.ttest_rel(pre.seen, pre.unseen)
print("Pre-stimulus alpha (8–12 Hz, −500 to −100 ms): seen %.2f vs unseen %.2f dB, t(%d)=%.2f, p=%.4f, %d/%d lower before seen" %
      (pre.seen.mean(), pre.unseen.mean(), N-1, t, p, (pre.seen < pre.unseen).sum(), N))


ERP spatio-temporal clusters (seen − unseen):
     sign  start_ms  end_ms  n_channels      p                                             channels
negative       133     797          85 0.0005 CPP3,PPO3,PPO5,PO7,PO9h,PO9,I1,OI1,O1,POO5,PO3h,PPO1
positive       141     797          83 0.0005 Cz,CCPz,CPz,CPPz,P1,CPP3h,CPP3,PPO3,PPO5,PO7,O1,POO5


              component     window  seen µV  unseen µV   t(30)      p  Cohen's dz  subjects in expected direction /31  p (Bonferroni, 2 tests)
VAN (occipito-temporal) 180-280 ms  -1.2613    -0.5723 -2.5360 0.0167     -0.4555                                  23                   0.0334
  P3b (centro-parietal) 350-650 ms   4.3985     1.8221  9.2852 0.0000      1.6677                                  30                   0.0000


TF clusters:
 sign    freq        time      p
   +  4-9 Hz  109-391 ms 0.0065
   − 4-30 Hz 281-1000 ms 0.0005


Decoding clusters:
  start_ms  end_ms      p
      156    1000 0.0005
Decoding peak AUC 0.649 ± 0.012 at 375 ms


Pre-stimulus alpha (8–12 Hz, −500 to −100 ms): seen 20.61 vs unseen 20.81 dB, t(30)=-2.06, p=0.0478, 21/31 lower before seen


## 3. 稳健性检验
- **时间泛化显著范围**："45%" = 显著簇覆盖的 训练时间×测试时间 格点数 / 全部格点数。
- **对比度匹配**：每位被试内，按对比度十分位，在每个区间内随机抽取等量的看见/没看见试次；重复 100 次（随机种子 1），每位被试取 100 次的平均差值 → 每人一个值 → 单样本 t 检验（N = 31）。
- **单试次回归**：幅值 ~ seen（0/1）+ contrast（z）+ trial（z），每位被试分别拟合，系数在组水平检验。
- **贝叶斯因子**：没看见 vs 捕获试次的 P3b，JZS 先验（Cauchy r = 0.707）。

In [4]:
# 回应评审：(a) 时间泛化显著范围 (b) 重复抽样的对比度匹配 + 含试次序号的回归 (c) unseen vs catch 的贝叶斯因子
import mne, numpy as np, pandas as pd, statsmodels.formula.api as smf, pingouin as pg
from scipy import stats
mne.set_log_level("ERROR")
G = np.load("group_data.npz"); dt = G["dec_times"]*1000
# (a) temporal generalization significant extent
Tg, clu, cp, _ = mne.stats.permutation_cluster_1samp_test(G["GEN"] - .5, n_permutations=1000, tail=1, seed=0, out_type="mask")
for c, p in zip(clu, cp):
    if p < .05:
        m = np.zeros_like(G["GEN"][0], bool); m[c if not isinstance(c, tuple) else c] = True
        tr, te = np.where(m)
        print(f"TG cluster p={p:.4f}: train {dt[tr].min():.0f}–{dt[tr].max():.0f} ms, test {dt[te].min():.0f}–{dt[te].max():.0f} ms, {m.mean()*100:.1f}% of matrix")
GEN = G["GEN"].mean(0); hi = GEN > 0.58; tr, te = np.where(hi); print(f"High-performance region (AUC>0.58): train {dt[tr].min():.0f}–{dt[tr].max():.0f}, test {dt[te].min():.0f}–{dt[te].max():.0f}")
# (b) contrast control: 100 random draws; regression with trial number
beh = pd.read_csv("results_behavior_qc.csv"); subs = beh.loc[beh.exclude_reason.isna() | (beh.exclude_reason == ""), "sub"].tolist()
roi = {"P3b": (["CPz","CPPz","Pz","P1","P2"], 0.35, 0.65), "VAN": (["PO7","PO8","P7","P8","O1","O2"], 0.18, 0.28)}
rng = np.random.default_rng(1); R = []
for s in subs:
    e = mne.read_epochs(f"{D}/compact/{s}_task-nocue_desc-128Hz_epo.fif")["stimulus=='present'"].crop(-0.2, 1.0)
    md = e.metadata.reset_index(drop=True); md["bin"] = pd.qcut(md.contrast.rank(method="first"), 10, labels=False)
    amps = {k: e.copy().pick(ch).crop(t0, t1).get_data().mean((1, 2))*1e6 for k, (ch, t0, t1) in roi.items()}
    seen = (md.condition == "seen").values; row = dict(sub=s)
    diffs = {k: [] for k in roi}; nmatch = []; cdiff = []
    for it in range(100):
        keep_s, keep_u = [], []
        for _, g in md.groupby("bin"):
            si, ui = g.index[g.condition == "seen"].values, g.index[g.condition == "unseen"].values; k = min(len(si), len(ui))
            keep_s += list(rng.choice(si, k, replace=False)); keep_u += list(rng.choice(ui, k, replace=False))
        nmatch.append(len(keep_s)); cdiff.append(md.contrast[keep_s].mean() - md.contrast[keep_u].mean())
        for k in roi: diffs[k].append(amps[k][keep_s].mean() - amps[k][keep_u].mean())
    row["n_matched"] = np.mean(nmatch); row["contrast_diff_matched"] = np.mean(cdiff)
    for k in roi:
        row[f"{k}_matched"] = np.mean(diffs[k])
        df = pd.DataFrame(dict(amp=amps[k], seen=seen.astype(int), contrast=(md.contrast-md.contrast.mean())/md.contrast.std(),
                               trial=(md.trial-md.trial.mean())/md.trial.std()))
        f = smf.ols("amp ~ seen + contrast + trial", df).fit()
        row[f"{k}_b_seen"] = f.params["seen"]; row[f"{k}_b_contrast"] = f.params["contrast"]; row[f"{k}_b_trial"] = f.params["trial"]
    R.append(row)
R = pd.DataFrame(R); R.to_csv("results_contrast_control_v2.csv", index=False); N = len(R)
t, p = stats.ttest_1samp(R.contrast_diff_matched, 0)
print(f"\nN={N}; matched trials/condition mean {R.n_matched.median():.0f} (median across subjects); residual contrast diff {R.contrast_diff_matched.mean():.5f}, t={t:.2f}, p={p:.3f}")
for col in [c for c in R.columns if "_" in c and c not in ("n_matched","contrast_diff_matched")]:
    t, p = stats.ttest_1samp(R[col], 0); print(f"{col:16s} mean {R[col].mean():6.3f} t({N-1})={t:6.2f} p={p:.4f}")
# (c) unseen vs catch Bayes factor (JZS, paired)
E = {k: mne.read_evokeds(f"erp_{k}-ave.fif") for k in ["unseen", "catch"]}
f = lambda evs: np.array([e.copy().pick(roi["P3b"][0]).crop(.35, .65).data.mean()*1e6 for e in evs])
u, c = f(E["unseen"]), f(E["catch"]); res = pg.ttest(u, c, paired=True)
print("\nUnseen vs catch P3b:", res[["T", "p_val", "BF10"]].to_string() if "p_val" in res else res.to_string())


TG cluster p=0.0010: train 125–1000 ms, test 109–1000 ms, 45.0% of matrix
High-performance region (AUC>0.58): train 250–797, test 250–797



N=31; matched trials/condition mean 134 (median across subjects); residual contrast diff 0.00003, t=1.89, p=0.068
P3b_matched      mean  2.469 t(30)=  8.74 p=0.0000
P3b_b_seen       mean  2.442 t(30)=  9.36 p=0.0000
P3b_b_contrast   mean  0.287 t(30)=  3.92 p=0.0005
P3b_b_trial      mean  0.101 t(30)=  0.99 p=0.3314
VAN_matched      mean -0.716 t(30)= -2.62 p=0.0138
VAN_b_seen       mean -0.666 t(30)= -2.42 p=0.0216
VAN_b_contrast   mean  0.069 t(30)=  0.88 p=0.3850
VAN_b_trial      mean -0.141 t(30)= -1.40 p=0.1712



Unseen vs catch P3b:                T     p_val BF10
T_test  0.307503  0.760585  0.2


## 4. 时间分块交叉验证（敏感性分析）
每个测试折 = 实验中连续的五分之一；由于部分时间块只含一类试次，先汇总各折的样本外决策值，再计算 AUC。同时报告常规 StratifiedKFold 测试折在时间上的跨度。

In [5]:
# 敏感性分析：严格按时间分块的交叉验证（KFold，不打乱；每个测试折 = 一段连续的实验时间）
import mne, numpy as np, pandas as pd
from scipy import stats
from sklearn.pipeline import make_pipeline; from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression; from sklearn.model_selection import KFold, StratifiedKFold
from mne.decoding import SlidingEstimator, cross_val_multiscore
mne.set_log_level("ERROR")
beh = pd.read_csv("results_behavior_qc.csv"); subs = beh.loc[beh.exclude_reason.isna() | (beh.exclude_reason == ""), "sub"].tolist()
AUCb = []; order_ok = []; skf_info = []
for s in subs:
    e = mne.read_epochs(f"{D}/compact/{s}_task-nocue_desc-128Hz_epo.fif")["stimulus=='present'"].crop(-0.2, 1.0).resample(64)
    md = e.metadata; order_ok.append(bool(np.all(np.diff(md.trial.values) > 0)))         # epochs 按实验顺序排列？
    X, y = e.get_data(), (md.condition == "seen").values.astype(int)
    # 记录常规 StratifiedKFold 的测试折在时间上是否连续
    for tr, te in StratifiedKFold(5).split(X, y):
        skf_info.append(dict(sub=s, span=(te.max()-te.min()+1)/len(y), contiguous=len(te)==te.max()-te.min()+1))
    clf = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
    # 时间分块的折中可能只有一类试次 → 汇总各折的样本外决策值，再统一计算 AUC
    from sklearn.metrics import roc_auc_score
    dec = np.zeros((len(y), X.shape[2]))
    for tr, te in KFold(5, shuffle=False).split(X):
        se = SlidingEstimator(clf, n_jobs=8).fit(X[tr], y[tr]); dec[te] = se.decision_function(X[te])
    AUCb.append(np.array([roc_auc_score(y, dec[:, t]) for t in range(X.shape[2])]))
    times = e.times*1000
A = np.array(AUCb); G = np.load("group_data.npz"); A0 = G["AUC"]
print("epochs in chronological order for all subjects:", all(order_ok))
si = pd.DataFrame(skf_info); print(f"StratifiedKFold test folds strictly contiguous: {si.contiguous.mean()*100:.0f}% of folds; median span {si.span.median()*100:.0f}% of session")
T, clu, cp, _ = mne.stats.permutation_cluster_1samp_test(A - .5, n_permutations=2000, tail=1, seed=0, out_type="mask")
for c, p in zip(clu, cp):
    mk = np.zeros(len(times), bool); mk[c[0] if isinstance(c, tuple) else c] = True; c = mk
    if p < .05: print(f"blocked-CV cluster {times[c][0]:.0f}–{times[c][-1]:.0f} ms, p={p:.4f}")
m, m0 = A.mean(0), A0.mean(0)
print(f"peak AUC blocked {m.max():.3f} at {times[m.argmax()]:.0f} ms vs stratified {m0.max():.3f} at {times[m0.argmax()]:.0f} ms")
b = A[:, times < 0].mean(1); t, p = stats.ttest_1samp(b, .5); print(f"baseline AUC blocked {b.mean():.3f}, t(30)={t:.2f}, p={p:.3f}")
print(f"correlation of group time courses r = {np.corrcoef(m, m0)[0,1]:.3f}; mean difference (stratified - blocked) {np.mean(m0-m):.4f}")
np.save("auc_blocked.npy", A)
# ---- 对比度匹配细节
R = pd.read_csv("results_contrast_control_v2.csv"); d = R.contrast_diff_matched; mn = beh.set_index("sub").loc[R["sub"], ["n_seen", "n_unseen"]].min(axis=1).values
ci = stats.t.interval(.95, len(d)-1, d.mean(), d.std(ddof=1)/np.sqrt(len(d)))
raw = beh.set_index("sub").loc[R["sub"]]; rd = raw.contrast_seen - raw.contrast_unseen
print(f"\nresidual contrast diff after matching: mean {d.mean():.6f}, 95% CI [{ci[0]:.6f}, {ci[1]:.6f}]; before matching mean {rd.mean():.5f} → reduction {100*(1-d.mean()/rd.mean()):.1f}%")
print(f"matched trials per condition: median {R.n_matched.median():.0f}, range {R.n_matched.min():.0f}–{R.n_matched.max():.0f}; % of smaller condition median {np.median(R.n_matched/mn*100):.0f}%, range {np.min(R.n_matched/mn*100):.0f}–{np.max(R.n_matched/mn*100):.0f}%")


epochs in chronological order for all subjects: True
StratifiedKFold test folds strictly contiguous: 3% of folds; median span 23% of session
blocked-CV cluster 172–828 ms, p=0.0005
peak AUC blocked 0.627 at 422 ms vs stratified 0.649 at 375 ms
baseline AUC blocked 0.477, t(30)=-2.97, p=0.006
correlation of group time courses r = 0.996; mean difference (stratified - blocked) 0.0214

residual contrast diff after matching: mean 0.000034, 95% CI [-0.000003, 0.000070]; before matching mean 0.00111 → reduction 97.0%
matched trials per condition: median 134, range 31–159; % of smaller condition median 88%, range 62–100%


## 5. 核对单元
(a) 检查二维聚类检验确实使用了二维邻接：显著簇在 频率×时间 / 训练×测试 平面上应为连通区域。
(b) 从输出文件中打印论文报告的每个关键数字。

In [6]:
import numpy as np, pandas as pd, mne
from scipy import ndimage, stats
G = np.load("group_data.npz"); N = G["AUC"].shape[0]
def components(mask): return ndimage.label(mask)[1]           # 4-邻接下的连通区域数
for name, X, thr, tail, nperm in [("time-frequency", G["TFs"] - G["TFu"], stats.t.ppf(0.995, N-1), 0, 2000),
                                  ("temporal generalization", G["GEN"] - .5, None, 1, 1000)]:
    T, clu, cp, _ = mne.stats.permutation_cluster_1samp_test(X, n_permutations=nperm, tail=tail, seed=0, threshold=thr, out_type="mask", verbose=False)
    for c, p in zip(clu, cp):
        if p < .05: print(f"{name}: significant cluster p = {p:.4f}, shape {c.shape}, connected components = {components(c)}, coverage = {c.mean()*100:.1f}%")
q = pd.read_csv("results_behavior_qc.csv"); inc = q[q.included]
print(f"\nN included = {len(inc)} / {len(q)}; excluded: {q.loc[~q.included, ['sub','exclude_reason']].values.tolist()}")
print(f"hit {inc.hit_rate.mean()*100:.1f}% ± {inc.hit_rate.std()*100:.1f}; FA {inc.fa_rate.mean()*100:.1f}% ± {inc.fa_rate.std()*100:.1f}; contrast {inc.contrast_seen.mean():.4f} vs {inc.contrast_unseen.mean():.4f}, seen>unseen in {(inc.contrast_seen>inc.contrast_unseen).sum()}/{len(inc)}")
print("\nROI tests:"); print(pd.read_csv("results_erp_roi.csv").to_string(index=False))
print("\nERP clusters:"); print(pd.read_csv("results_erp_clusters.csv")[["sign","start_ms","end_ms","n_channels","p"]].to_string(index=False))
print("\nTF clusters:"); print(pd.read_csv("results_tf_clusters.csv").to_string(index=False))
print("\nDecoding clusters:"); print(pd.read_csv("results_decoding_clusters.csv").to_string(index=False))
r = pd.read_csv("results_contrast_control_v2.csv")
for col in ["P3b_matched", "VAN_matched", "P3b_b_seen", "VAN_b_seen", "P3b_b_contrast", "VAN_b_contrast", "P3b_b_trial", "VAN_b_trial"]:
    t, p = stats.ttest_1samp(r[col], 0); print(f"{col:15s} mean {r[col].mean():6.2f}  t({len(r)-1}) = {t:6.2f}  p = {p:.4f}")

time-frequency: significant cluster p = 0.0065, shape (27, 84), connected components = 1, coverage = 3.8%
time-frequency: significant cluster p = 0.0005, shape (27, 84), connected components = 1, coverage = 28.2%


temporal generalization: significant cluster p = 0.0010, shape (78, 78), connected components = 1, coverage = 45.0%

N included = 31 / 35; excluded: [['sub-08', 'interp>10%;'], ['sub-16', '<30 trials;'], ['sub-20', '<30 trials;'], ['sub-34', 'interp>10%;']]
hit 42.6% ± 11.4; FA 6.6% ± 7.3; contrast 0.0337 vs 0.0326, seen>unseen in 29/31

ROI tests:
              component     window  seen µV  unseen µV   t(30)      p  Cohen's dz  subjects in expected direction /31  p (Bonferroni, 2 tests)
VAN (occipito-temporal) 180-280 ms  -1.2613    -0.5723 -2.5360 0.0167     -0.4555                                  23                   0.0334
  P3b (centro-parietal) 350-650 ms   4.3985     1.8221  9.2852 0.0000      1.6677                                  30                   0.0000

ERP clusters:
    sign  start_ms  end_ms  n_channels      p
negative       133     797          85 0.0005
positive       141     797          83 0.0005

TF clusters:
sign    freq        time      p
   +  4-9 Hz  109-391